# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

In [2]:
# Quick look at the folder structure so we know where the data lives
import os
print(os.listdir('data'))

['.gitkeep', 'AviationData.csv', 'AviationData_Cleaned.csv', 'aviation_accidents_cleaned.csv', 'USState_Codes.csv']


### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
# The file contains a few non-UTF8 characters, so we use latin-1 encoding
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

print('Shape:', df.shape)
display(df.head(3))
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False))
print('\nData types:')
print(df.dtypes)
display(df.describe())

Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007



Missing values per column:
Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date  

,Event.Date,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,88889,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1999-09-17 17:13:39.354475,1.146585,0.647855,0.279881,0.357061,5.325440
min,1948-10-24 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1989-01-15 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1998-07-18 00:00:00,1.000000,0.000000,0.000000,0.000000,1.000000
75%,2009-07-01 00:00:00,1.000000,0.000000,0.000000,0.000000,2.000000
max,2022-12-29 00:00:00,8.000000,349.000000,161.000000,380.000000,699.000000
std,NaN,0.446510,5.485960,1.544084,2.235625,27.913634


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
# 1. keep 1983 onwards
df = df[df['Event.Date'].dt.year >= 1983].copy()

# 2. professional builds only
df = df[df['Amateur.Built'] == 'No'].copy()

# Make names come in mixed case (Cessna / CESSNA) - standardise first so the helicopter list below works
df['Make'] = df['Make'].str.upper().str.strip()

# 3. impute missing Aircraft.Category as 'Airplane' where it is very likely
helicopter_makes = ['BELL', 'HUGHES', 'ROBINSON', 'SCHWEIZER', 'ENSTROM', 'HILLER', 'SIKORSKY',
                    'EUROCOPTER', 'MD HELICOPTERS', 'AGUSTA', 'KAMAN', 'BRANTLY']
likely_airplane = (
    df['Aircraft.Category'].isna()
    & df['Engine.Type'].isin(['Reciprocating', 'Turbo Prop', 'Turbo Fan', 'Turbo Jet'])
    & (df['Number.of.Engines'] >= 1)
    & ~df['Make'].isin(helicopter_makes)
)
print('Categories imputed as Airplane:', likely_airplane.sum())
df.loc[likely_airplane, 'Aircraft.Category'] = 'Airplane'

df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print('Rows remaining:', len(df))

Categories imputed as Airplane: 43921
Rows remaining: 65368


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [5]:
# Estimate the number of people on board = fatal + serious + minor + uninjured
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# If ALL four columns are empty we know nothing about the people on board -> drop the row
df = df.dropna(subset=injury_cols, how='all').copy()
# If only some are empty, a blank means "nobody in that category" -> fill with 0
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# A flight with 0 people on board (e.g. a ground incident) has no meaningful injury fraction
df = df[df['Total.Passengers'] > 0].copy()

# Derived metric: share of people on board who were killed or seriously injured
df['FatalSerious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(len(df), 'rows left')
display(df[['Total.Passengers', 'FatalSerious.Fraction']].describe())

64453 rows left


,Total.Passengers,FatalSerious.Fraction
count,64453.000000,64453.000000
mean,7.022807,0.266004
std,30.771597,0.426358
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,0.500000
max,699.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [6]:
print(df['Aircraft.damage'].value_counts(dropna=False))

# 'Unknown' and missing damage can't be used to say if the plane was destroyed -> drop them
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()

# Derived column: 1 if the aircraft was destroyed, 0 otherwise
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
print('\nShare destroyed:', round(df['Destroyed'].mean(), 3))

Aircraft.damage
Substantial    47724
Destroyed      12903
Minor           1946
NaN             1806
Unknown           74
Name: count, dtype: int64

Share destroyed: 0.206


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [7]:
# See which spellings exist for the big manufacturers
for key in ['AIRBUS', 'BOMBARDIER', 'DOUGLAS', 'HAVILLAND', 'CIRRUS', 'GRUMMAN', 'AIR TRACTOR', 'ERCOUPE', 'BRITISH']:
    print(key, '->', df.loc[df['Make'].str.contains(key), 'Make'].value_counts().to_dict())

AIRBUS -> {'AIRBUS INDUSTRIE': 62, 'AIRBUS': 62, 'AIRBUS INDUSTRIES': 1}
BOMBARDIER -> {'BOMBARDIER': 43, 'BOMBARDIER INC': 33, 'BOMBARDIER, INC.': 13, 'BOMBARDIER AEROSPACE, INC.': 1, 'BOMBARDIER LEARJET CORP.': 1, 'BOMBARDIER CANADAIR': 1}
DOUGLAS -> {'MCDONNELL DOUGLAS': 269, 'DOUGLAS': 192, 'MCDONNELL DOUGLAS AIRCRAFT CO': 6, 'DOUGLAS/BASLER': 1, 'MCDONNELL-DOUGLAS': 1, 'EDWARDS DOUGLAS L': 1, 'MCDONNELL DOUGLAS CORPORATION': 1}
HAVILLAND -> {'DE HAVILLAND': 348, 'DEHAVILLAND': 96, 'BOEING - CANADA (DE HAVILLAND)': 2, 'DEHAVILLAND CANADA': 1}
CIRRUS -> {'CIRRUS DESIGN CORP': 219, 'CIRRUS': 137, 'CIRRUS DESIGN CORP.': 65, 'CIRRUS DESIGN CORPORATION': 13, 'CIRRUS DESIGN': 5}
GRUMMAN -> {'GRUMMAN': 1049, 'GRUMMAN AMERICAN': 209, 'GRUMMAN-SCHWEIZER': 126, 'GRUMMAN ACFT ENG COR-SCHWEIZER': 58, 'GRUMMAN AMERICAN AVN. CORP.': 49, 'GRUMMAN AMERICAN AVIATION': 2, 'GRUMMAN ACFT ENG': 2, 'GRUMMAN AIRCRAFT ENG CORP': 2, 'GRUMMAN SCHWEIZER': 2, 'GRUMMAN AMERICAN AVIATION CORP': 1, 'GRUMMAN AIRC

In [8]:
make_map = {
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'BOMBARDIER INC': 'BOMBARDIER', 'BOMBARDIER, INC.': 'BOMBARDIER', 'CANADAIR': 'BOMBARDIER',
    'MCDONNELL DOUGLAS AIRCRAFT CO': 'MCDONNELL DOUGLAS',
    'DEHAVILLAND': 'DE HAVILLAND',
    'AIR TRACTOR INC': 'AIR TRACTOR',
    'CIRRUS DESIGN CORP': 'CIRRUS',
    'GRUMMAN AMERICAN': 'GRUMMAN',
    'ERCOUPE (ENG & RESEARCH CORP.)': 'ERCOUPE',
    'SAAB-SCANIA AB (SAAB)': 'SAAB',
    'AVIAT AIRCRAFT INC': 'AVIAT',
    'GRUMMAN-SCHWEIZER': 'GRUMMAN-SCHWEIZER', 'GRUMMAN ACFT ENG COR-SCHWEIZER': 'GRUMMAN-SCHWEIZER',
}
df['Make'] = df['Make'].replace(make_map)

make_counts = df['Make'].value_counts()
keep_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(keep_makes)].copy()
print(len(keep_makes), 'makes kept;', len(df), 'rows left')
print(df['Make'].value_counts().head(15))

56 makes kept; 57980 rows left
Make
CESSNA            25010
PIPER             13726
BEECH              4901
GRUMMAN            1258
MOONEY             1246
BOEING             1174
BELLANCA            964
AIR TRACTOR         874
AERONCA             604
MAULE               565
CHAMPION            500
DE HAVILLAND        444
STINSON             420
AERO COMMANDER      393
LUSCOMBE            384
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [9]:
# 1. Drop missing models and standardise the text
df = df.dropna(subset=['Model']).copy()
df['Model'] = df['Model'].str.upper().str.strip()

# 2. Are model labels unique to one make?  (e.g. '172' only for Cessna?)
makes_per_model = df.groupby('Model')['Make'].nunique()
print('Model labels used by more than one make:', (makes_per_model > 1).sum(), 'of', len(makes_per_model))
print(makes_per_model[makes_per_model > 1].sort_values(ascending=False).head(8))

# 3. They are not unique, so build a unique identifier = Make + Model
df['Plane'] = df['Make'] + ' ' + df['Model']
print('\nUnique plane types:', df['Plane'].nunique())

Model labels used by more than one make: 236 of 4162
Model
500      5
112      4
500-S    4
35A      4
7AC      4
7EC      4
7ECA     4
8KCAB    4
Name: Make, dtype: int64

Unique plane types: 4457


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [10]:
# Inspect each column first
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False).head(12), '\n')

Engine.Type
Reciprocating      51336
Turbo Prop          2510
NaN                 2376
Turbo Fan           1294
Turbo Jet            370
Unknown               71
Turbo Shaft            9
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

Weather.Condition
VMC    51240
IMC     4615
NaN     1512
UNK      466
Unk      135
Name: count, dtype: int64 

Number.of.Engines
1.0    48029
2.0     7988
NaN     1391
4.0      278
3.0      278
0.0        4
Name: count, dtype: int64 

Purpose.of.flight
Personal               34364
Instructional           7936
Unknown                 3609
Aerial Application      3221
Business                2873
NaN                     2464
Positioning              973
Ferry                    522
Other Work Use           504
Aerial Observation       369
Executive/corporate      326
Public Aircraft          262
Name: count, dtype: int64 

Broad.phase.of.flight
NaN            14676
Landing        11906
Takeoff         8870
Cruise          7080


In [11]:
df['Engine.Type'] = df['Engine.Type'].replace(['Unknown', 'UNK', 'NONE', 'LR'], np.nan)
df['Weather.Condition'] = df['Weather.Condition'].replace(['UNK', 'Unk'], np.nan)
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Unknown', np.nan)
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace(['Unknown', 'Other'], np.nan)

for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines']:
    print(df[col].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating      51336
Turbo Prop          2510
NaN                 2448
Turbo Fan           1294
Turbo Jet            370
Turbo Shaft            9
Geared Turbofan        1
Name: count, dtype: int64 

Weather.Condition
VMC    51240
IMC     4615
NaN     2113
Name: count, dtype: int64 

Number.of.Engines
1.0    48029
2.0     7988
NaN     1395
4.0      278
3.0      278
Name: count, dtype: int64 



### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [12]:
na_share = df.isna().mean().sort_values(ascending=False)
print(na_share.head(14))

# Drop columns that are more than 50% empty
too_empty = na_share[na_share > 0.5].index.tolist()
print('\nDropping:', too_empty)
df = df.drop(columns=too_empty)

# Also drop identifier/free-text columns we will never analyse
df = df.drop(columns=['Accident.Number', 'Registration.Number', 'Report.Status', 'Publication.Date'], errors='ignore')


Schedule                 0.884246
Air.carrier              0.833408
FAR.Description          0.709305
Longitude                0.632159
Latitude                 0.632056
Airport.Code             0.379140
Airport.Name             0.350193
Broad.phase.of.flight    0.260368
Publication.Date         0.178167
Purpose.of.flight        0.104765
Report.Status            0.049821
Engine.Type              0.042230
Weather.Condition        0.036451
Number.of.Engines        0.024065
dtype: float64

Dropping: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [13]:
df.to_csv('data/AviationData_Cleaned.csv', index=False)
print('Saved', df.shape)


Saved (57968, 26)
